<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/03-rag/01-embeddings-retrieval.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Embedding 与检索

**目标：** 选好 embedding、搭起向量搜索，并踩中那些经典的相似度陷阱。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 环境准备

每个 notebook 都是自包含的，接下来的两个单元格会把它从零搭起来：

1. **安装依赖。** `aien` 包（本仓库）带着共享的环境准备 helper，并拉入 `groq` 客户端；本篇还会用到 `requests`、`sentence-transformers`、`numpy`。
2. **载入你的 API key。** 免费 key 在 [console.groq.com](https://console.groq.com/) 领（无需信用卡）。在 Colab 里通过左侧边栏的**钥匙图标** → **Add new secret** 添加，名称必须严格写成 `GROQ_API_KEY`，粘贴值，并把 **Notebook access** 打开。本地运行的话，则把 `GROQ_API_KEY` 设成环境变量。

（完整走查和选模型的指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main" requests sentence-transformers numpy

In [ ]:
from aien import setup

# 载入 GROQ_API_KEY（Colab Secrets 或本地环境变量）并返回一个可直接
# 使用的 Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用
# 报 404，就列出可用模型 —— 见 00-setup/00-environment.ipynb。
client, MODEL = setup()

## 问题：你的用户不会按关键词打字

要搜索文档，你先把它们切成 **chunk**——段落大小的片段，每块是一个可检索单元（下一个单元格会解释为什么这一步不可避免）。有了 chunk，下面就是 embedding 要解决的问题。

有人问*「连接是怎么建立起来的？」*，而回答这个问题的 chunk 写的是「三次握手」，从头到尾没出现过「建立」这两个字。关键词搜索就会漏掉它。这个缺口——用户用自己的话提问、文档用自己的词表述——正是 **embedding** 所能弥合的：它把文本映射成向量，让*含义相近*的东西*落在附近*，所以一句同义改写仍然能找到它的答案。

这就是本篇的目标：把 chunk 变成向量、检索它们，并把命中的前几名接进一次生成回答。但 embedding 是用代价换来语义匹配的，后半篇讲的就是这个代价：三个必然会在你的生产日志里出现的陷阱。要一路记住的那条承重规则是：**生成出来的答案，永远只能和检索取回的 chunk 一样好。** 检索做错了，任何 prompt 微调都救不了你。

## 环境准备：语料与 chunk

**为什么这里没法跳过分块。** 在 notebook 00 里，「文档」都是单句：一篇文档就是一个可检索单元，没什么可切的。真实文档不是这样。本语料里一篇 RFC 有上万 token，而下面的 embedding 模型（`all-MiniLM-L6-v2`）只读取你交给它的内容的前约 **256 个 token**，剩下的会被它默默忽略。把一整篇 RFC 拿去 embed，你得到的只是它开头那些套话的向量，别的一无所有，检索也就废了。所以每篇文档*必须*先被切成段落大小的 **chunk** 再做 embedding。这就是分块背后的强制力，也是本篇无法回避它的原因。

我们用一个合理的 **section-aware 切分器**（按编号标题切分，限制大小），并在此把它当成既定条件——够用就好，只要能拿到可检索的 chunk。*怎么*切才切得好、这个选择对检索质量的影响有多大，是它自己的一篇 notebook（**03**），特意排在本篇之后：在你亲眼看着检索在某个分块策略上成功和失败之前，你没法评判它——而这正是你接下来要做的事。

下载那个单元格被原样重复了一遍（本仓库每个 notebook 都自包含），分块器则是 notebook 03 逐步搭出来的那个的精简复制。


In [ ]:
# 下载共享语料：十篇 IETF RFC，缓存到 data/rfc/。
# 本仓库里每个需要这份语料的 notebook 都包含这个单元格 ——
# 自包含优先于 DRY，好让每个 notebook 都能自己从头跑到尾。
import os
import requests

RFC_NUMBERS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
RFC_TITLES = {
    791: 'IP', 793: 'TCP', 1035: 'DNS', 2616: 'HTTP/1.1', 4271: 'BGP',
    5321: 'SMTP', 6455: 'WebSocket', 6749: 'OAuth 2.0', 7540: 'HTTP/2',
    9110: 'HTTP Semantics',
}
DATA_DIR = 'data/rfc'
os.makedirs(DATA_DIR, exist_ok=True)

corpus = {}  # rfc 编号 -> 原始文本
for num in RFC_NUMBERS:
    path = os.path.join(DATA_DIR, f'rfc{num}.txt')
    if not os.path.exists(path):  # 已缓存：重跑时跳过下载
        resp = requests.get(f'https://www.rfc-editor.org/rfc/rfc{num}.txt', timeout=30)
        resp.raise_for_status()
        with open(path, 'w') as f:
            f.write(resp.text)
    with open(path) as f:
        corpus[num] = f.read()

for num in RFC_NUMBERS:
    print(f'RFC {num:>4}  {RFC_TITLES[num]:<15} {len(corpus[num]):>9,} chars')

In [ ]:
# 清洗 + 按 section 切分的精简复制。notebook 03（分块）
# 才是把它逐步搭起来并解释其权衡的地方；这里只是一个既定条件。
import re

def clean_rfc(text):
    text = text.replace('\f', '\n')
    lines = [l for l in text.split('\n')
             if not re.match(r'^.*\[Page \d+\]\s*$', l)        # 页脚
             and not re.match(r'^RFC \d+\s+.*\S+ \d{4}\s*$', l)]  # 页眉
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines)).strip()

HEADING_RE = re.compile(r'^\d+(?:\.\d+)*\.?\s+\S', re.MULTILINE)

def chunk_paragraphs(text, max_chars=2400):
    paras = [p for p in text.split('\n\n') if p.strip()]
    chunks, cur = [], ''
    for p in paras:
        if cur and len(cur) + len(p) + 2 > max_chars:
            chunks.append(cur)
            cur = p
        else:
            cur = cur + '\n\n' + p if cur else p
    if cur:
        chunks.append(cur)
    return chunks

def chunk_sections(text, max_chars=2400):
    starts = [m.start() for m in HEADING_RE.finditer(text)]
    if not starts:
        return chunk_paragraphs(text, max_chars)
    chunks = [text[:starts[0]].strip()] if text[:starts[0]].strip() else []
    for a, b in zip(starts, starts[1:] + [len(text)]):
        sec = text[a:b].strip()
        if len(sec) <= max_chars:
            chunks.append(sec)
        else:  # 长 section：按段落打包，并把标题一起带上
            heading = sec.split('\n', 1)[0]
            for sub in chunk_paragraphs(sec, max_chars):
                chunks.append(sub if sub.startswith(heading) else heading + '\n' + sub)
    return [c for c in chunks if c]

chunk_texts, chunk_meta = [], []
for num in RFC_NUMBERS:
    for c in chunk_sections(clean_rfc(corpus[num])):
        chunk_texts.append(c)
        chunk_meta.append({'rfc': num, 'title': RFC_TITLES[num]})
print(f'{len(chunk_texts)} chunks across {len(RFC_NUMBERS)} RFCs')

## 给 chunk 做 embedding

我们用 sentence-transformers 的 `all-MiniLM-L6-v2`：它在本地运行、免费，而且 torch 的安装在 Colab 上能搞定。生产系统会用一个托管 embedding API（Voyage、OpenAI、Cohere），但你在这里学的机制完全一样，而且把 embedding 保持免费，意味着你的 API 预算能花在生成和评测上——那才是它真正为你买到东西的地方。

在免费的 Colab CPU 上，给几千个 chunk 做 embedding 要花几分钟；在 T4 GPU runtime 上只要几秒。两种都行。


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(chunk_texts, normalize_embeddings=True,
                             show_progress_bar=True)
print(embeddings.shape)  # (n_chunks, 384)

## 用 numpy 做暴力余弦搜索

向量都是归一化的，所以余弦相似度就是点积，而搜索整个语料就是一次矩阵乘向量。这就是整个向量数据库，大约十行代码。

在学习规模下（几千个 chunk），暴力搜索不是偷懒的捷径。它是*正确*的：精确、毫秒级、没有索引需要调、也不会失效过期。近似最近邻索引（HNSW、IVF）是为 1 亿向量的规模存在的——到那个量级精确搜索就不再划算，你开始用召回换延迟。


## 拿它检索：`search` 与 `show`

两个小 helper：`search(query, k)` 用同样的方式给查询做 embedding，与每个 chunk 做点积，返回前 k 个 `(score, index)`；`show` 把它们可读地打印出来。这就是整个检索层。先用三个*措辞得体*的问题确认它能用。读前 5 名时要连分数一起读，别只看标题。

In [ ]:
def search(query, k=5):
    """给查询做 embedding，对所有 chunk 做余弦排序，返回前 k 个 (score, index)。"""
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = embeddings @ q                       # 余弦：向量都已单位归一化
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), int(i)) for i in top]

def show(query, k=5):
    """跑一次检索，带分数打印前 k 个命中。"""
    print(f'Q: {query}')
    for score, i in search(query, k):
        first_line = chunk_texts[i].strip().split('\n')[0][:66]
        print(f'  {score:.3f}  RFC {chunk_meta[i]["rfc"]:>4} ({chunk_meta[i]["title"]})  {first_line}')
    print()

# 三个措辞得体的问题：语义搜索应该能稳稳命中。
show('How does the TCP three-way handshake work?')
show('What does the HTTP 404 status code mean?')
show('How does a client obtain an OAuth access token?')

这三个都干净命中：正确的 RFC 占据主导，给出答案的那一节浮到最上面。对于措辞得体的提问、面对这样干净的语料，语义搜索看起来很棒。

现在我们从三个角度把它弄坏。下面每个陷阱都是真实查询模式，必然出现在你的生产日志里，这里现场演示。先一览全貌，然后每个一节：

| # | 陷阱 | 症状 | 为什么会这样 | 最省事的修法 |
|---|---|---|---|---|
| 1 | **相似 ≠ 相关** | 主题接近但意图不对的 chunk 排到最前 | embedding 匹配的是含义，不是查询没有说出口的意图 | 用元数据限定范围；用已知上下文丰富查询 |
| 2 | **词汇不匹配** | 精确标识符（header、端口、opcode）无法可靠命中 | 384 个数保不住精确的 token；相近兄弟项互相撞车 | 在向量之外并上关键词搜索（notebook 02） |
| 3 | **分数不是置信度** | 一个无法回答的查询仍然给出很高的头名分数 | 最近邻总会返回*某个东西* | 用 reranker 或评测，而不是分数阈值 |

## 陷阱 1：相似 ≠ 相关

Embedding 度量的是*语义相似度*，不是*与你意图的相关性*。问一个关于连接超时的问题，也就是按 Web 开发者的想法理解为 HTTP 请求超时，看看它会返回什么。

In [ ]:
show('how long should I wait before timing out a connection?')

跑一下，注意 TCP 那部分材料（RFC 793、重传超时）排在最前或接近最前。那段文本*关于*超时、也*关于*连接，所以它在 embedding 空间里离得近，尽管一个在 HTTP 语境下提这个问题的用户想要的是 RFC 9110/2616 的材料。embedding 忠实地找到了相似的文本。相似不等于相关：相关性取决于查询没有说出来的意图。

修法很朴素，但有效：让用户能限定搜索范围（按元数据过滤，而我们正好有 `chunk_meta` 派这个用场），或者用你已知的上下文丰富查询（「在 HTTP 里，……」）。试试：


In [ ]:
show('in HTTP, how long should I wait before timing out a connection?')

## 陷阱 2：词汇不匹配，关键词搜索在此胜出

Embedding 把文本压缩成 384 个数。精确标识符（端口号、header 字段名、opcode）在这种压缩下保不住自己的身份：`Sec-WebSocket-Accept` 和 `Sec-WebSocket-Key` 在 embedding 空间里几乎叠在一起，可它们是规则不同的两个字段。

拿两个精确标识符查询，把向量搜索和一个笨笨的子串 grep 比一比：


In [ ]:
def grep(term, k=5):
    hits = [i for i, t in enumerate(chunk_texts) if term.lower() in t.lower()]
    print(f'grep {term!r}: {len(hits)} chunks contain it, e.g.')
    for i in hits[:k]:
        first_line = chunk_texts[i].strip().split('\n')[0][:72]
        print(f'         RFC {chunk_meta[i]["rfc"]:>4} ({chunk_meta[i]["title"]})  {first_line}')
    print()

show('Retry-After')
grep('Retry-After')

show('Sec-WebSocket-Accept')
grep('Sec-WebSocket-Accept')

跑一下再对比。grep 简单直接、完美命中：每个结果都字面包含那个标识符。向量搜索捞回的是*主题上相近*的文本（别的 header 字段、别的握手材料），可能命中、也可能不命中那些真正定义 `Retry-After` 或 `Sec-WebSocket-Accept` 的 chunk（而不是在 embedding 空间里几乎压在它上面的兄弟项 `Sec-WebSocket-Key`）。对于「查询本身就是精确标识符」的场景，一项 1970 年代的技术打败了 embedding 模型。

这不是在反对 embedding，而是在说明没有哪个工具能独占上风。notebook 02 把两者结合起来。

## 陷阱 3：分数不是校准过的置信度

很容易把余弦分数当成置信度：「如果头名低于 0.5，就说我不知道。」看看分数大小在不同查询之间怎么变，其中有一个查询的答案根本不在本语料里：


In [ ]:
for q in [
    'How does TCP recover from a lost segment?',        # 稳稳在语料内（RFC 793）
    'What port does SMTP use?',                          # 在语料内，答案很简短
    'How does Kubernetes schedule pods onto nodes?',     # 完全不在本语料里
]:
    top_score, i = search(q, 1)[0]
    print(f'{top_score:.3f}  top hit: RFC {chunk_meta[i]["rfc"]} ({chunk_meta[i]["title"]})  <- {q}')

跑一下，注意那个 Kubernetes 查询——从十篇网络 RFC 里完全无法回答——仍然拿到了和那些可回答问题大致同一区间的头名分数。最近邻搜索总会返回*某个东西*；这个分数说的是「这是最近的 chunk」，而不是「这个 chunk 回答了问题」。分数大小还会随查询长度和措辞漂移，所以在一个查询分布上调好的阈值，换到另一个分布上就会悄悄失灵。

如果你需要一个「这个到底相不相关？」的信号，你要么需要一个被训练来产生该信号的组件（reranker，见 notebook 02），要么需要一个能度量它的评测（本仓库第 04 节）。原始余弦数字只是一个排序工具，仅此而已。

## 从检索到生成：闭环

RAG 的最后一步很平凡：把前 k 个 chunk 连同引用标记塞进 prompt，让模型依据它们作答。


In [ ]:
def answer(question, k=5):
    hits = search(question, k)
    context = '\n\n'.join(
        f'[{n + 1}] (RFC {chunk_meta[i]["rfc"]}, {chunk_meta[i]["title"]})\n{chunk_texts[i][:1500]}'
        for n, (_, i) in enumerate(hits)
    )
    prompt = (
        'Answer the question using only the numbered passages below. '
        'Cite the passages you used like [1] or [2][3].\n\n'
        f'{context}\n\nQuestion: {question}'
    )
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=1024,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return resp.choices[0].message.content

print(answer('What does an HTTP 503 response mean, and which header can tell the client when to retry?'))

跑一下，把引用和检索到的 chunk 核对一遍。当检索把正确的段落放进 prompt 时，生成这一步就很轻松：模型很会读。

要内化的就是这种不对称：**答案只和命中的结果一样好。** 生成无法引用一个检索没取回的 chunk，而且它会拿*实际*给到它的东西做点什么——通常是很流畅的东西。所以本篇每一个陷阱都是迟早要发生的答案质量 bug，而 notebook 04 讲的正是从答案这一侧去诊断它们。


## 练习

1. **元数据过滤。** 给 `search()` 加一个 `rfc=` 参数，把结果限制在一篇 RFC 内（用 `chunk_meta` 把分数向量掩掉）。把陷阱 1 的超时查询过滤到 RFC 9110 再跑一遍，确认当用户能限定搜索范围时，意图问题就消失了。
2. **分块器 A/B。** 用 `chunk_paragraphs`（在上面那个环境准备单元格里定义）而不是 `chunk_sections` 重建索引，再跑一遍开头的三个 `show()` 查询。把两份前 5 名列表并排对比：同一份语料、同一个 embedding 模型、不同的分块。这是 notebook 03 整套论证的预演。分块是你手上最便宜的检索质量杠杆，而在这里你已经能感觉到它在推动结果。
3. **分数落差启发式。** 对陷阱 3 里的三个查询，分别计算头名和第 5 名之间的分数落差。作为语料外信号，这个落差比绝对分数更好吗？写一个 `maybe_unanswerable(query)` 函数，再找一个能骗过它的查询。
4. **查询扩展。** 检索之前，用模型（`MODEL`，一次便宜调用）把查询改写成三个同义说法，用这三个一起检索，再按最高分数合并结果。在陷阱 2 的标识符查询上测试。扩展能救回词汇不匹配吗，还是关键词搜索仍然赢？
